In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset,DataLoader

from tqdm import tqdm

SR = 22050
DURATION = 30
model_name = "cnn"

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kgg_key")
secret_value_1 = user_secrets.get_secret("kgg_user")


#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")

🚀 Using device: cpu
   Running on CPU

✅ Environment setup complete!


Your First Neural Network & CNNs!

* Learn PyTorch basics: Tensors, Dataset (custom loader for training), DataLoader.
* Convert audio to 2D/1D Mel-Spectrograms.
* Build a simple CNN (Convolutional Neural Network)/NN (Neural Network) to process the spectrograms.
* Implement training loop, loss, optimizer, and wandb logging.
* Train and evaluate your CNN/NN (Neural Network)model.

# Definition

## 1. Utility

In [10]:
def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    waveform, _sr_ = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if _sr_ != sr:
        resampler = torchaudio.transforms.Resample(_sr_, sr)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    # Trim or pad
    if waveform.shape[0] >= LENGTH:
        return waveform[:LENGTH],sr
    else:
        padding = LENGTH - waveform.shape[0]
        return torch.nn.functional.pad(waveform, (0, padding)), sr

def extract_paths_ids():
    test_paths = []
    count = 0
    root_to_test = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    test_csv = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
    for song_id,song_path in zip(test_csv['id'],test_csv['filename']):
        path = os.path.join(root_to_test,song_path)
        test_paths.append((song_id,path))
        count += 1
    print("Total paths (music files) : ", count)
    return test_paths

def genre_to_idx(targets):
    genre_to_id = {'blues':0, 'classical':1, 'country':2, 'disco':3, 'hiphop':4,'jazz':5, 'metal':6, 'pop':7, 'reggae':8, 'rock':9}
    y = torch.tensor([genre_to_id[g] for g in targets])

    return y

def idx_to_genre(targets):
    id_to_genre = {0:'blues', 1:'classical', 2:'country', 3:'disco', 4:'hiphop',5:'jazz', 6:'metal', 7:'pop', 8:'reggae', 9:'rock'}
    y = [id_to_genre[id] for id in targets]

    return y    

## 2. Dataset and DataLoader

In [16]:
class MelDataset(Dataset):
    def __init__(self,paths):  # paths : List[(path,label)]
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self,idx):
        waveform, sr = load_and_fix(self.paths[idx][1],sr=SR,duration=DURATION)
        return self.paths[idx][0],waveform

config = {
    "batch_size" : 128
}

mel_transform = T.MelSpectrogram(
    sample_rate = 22050,
    n_fft = 1024,
    n_mels=128
).to(device)

amplitude_to_db = T.AmplitudeToDB().to(device)

test_paths = extract_paths_ids()

test_dataset = MelDataset(test_paths)


test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

print(f"Size of Test Dataloader : {len(test_loader)}")
print("✅")

Total paths (music files) :  3020
Size of Test Dataloader : 24
✅


## 3. Model

In [13]:
class MelCNN(nn.Module):
    def __init__(self, num_classes):
        super(MelCNN, self).__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):

        x = x.unsqueeze(1)  # (B,1,128,1292)

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

## 5. Evaluation

In [23]:
def evaluation(model,test_loader):
    model.to(device)
    model.eval()
    all_ids = []
    all_pred = []
    with torch.no_grad():
        for ids,waveforms in tqdm(test_loader, desc="Evaluating"):
            waveforms = waveforms.to(device,non_blocking=True)
            
            mels = mel_transform(waveforms)
            mels = amplitude_to_db(mels)
            
            output = model(mels)

            probs = torch.softmax(output,dim=1) 
            predicted_y = torch.argmax(probs,dim=1)

            all_pred.append(predicted_y)
            all_ids.append(ids)
        all_pred = torch.cat(all_pred,dim=0).cpu().numpy()
        all_ids = torch.cat(all_ids,dim=0).cpu().numpy()
        
        print(all_pred.shape)
        print(all_ids.shape)

        return all_pred,all_ids

# Loading models

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.model_download("akashkumbhakar/cnn/pyTorch/default")
print("Path to model files:", path)

# Evaluation

In [24]:
model = MelCNN(10)
model.load_state_dict(torch.load("/kaggle/input/models/akashkumbhakar/cnn/pytorch/default/1/model.pt",map_location=torch.device(device)))
all_pred,all_ids = evaluation(model,test_loader)

Evaluating: 100%|██████████| 24/24 [05:02<00:00, 12.61s/it]

(3020,)
(3020,)


# Saving submission

In [27]:
sample_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv"
sample_sub = pd.read_csv(sample_dir)
print("✅ Sample Submission dataset loaded.")
sample_sub['genre']

✅ Sample Submission dataset loaded.


0            jazz
1           blues
2       classical
3             pop
4           disco
          ...    
3015          pop
3016        blues
3017         jazz
3018         rock
3019       reggae
Name: genre, Length: 3020, dtype: object

In [30]:
y_pred = idx_to_genre(all_pred)
y_pred[:5]

['pop', 'blues', 'disco', 'disco', 'country']

In [31]:
sample_sub['genre'] = y_pred

In [32]:
sample_sub['genre'].value_counts()

genre
disco        574
reggae       473
blues        453
hiphop       339
pop          277
metal        259
rock         258
country      231
jazz          94
classical     62
Name: count, dtype: int64

In [33]:
sample_sub.to_csv('submission.csv',index=False)
if os.path.exists("/kaggle/working/submission.csv"):
    print("✅ Submission.csv created successfully.")

✅ Submission.csv created successfully.
